# Stochastic Calculus from Zero · Episode 7: Random walks and the Markov property

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Random walks are simulated with a fixed seed; Sections 7-8 use the coin world (share 100, ×1.2 or ×0.8 per toss, heads 60%), made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. A random walk · 4. Independent pieces add their variances · 5. Squared steps, exactly · 6. A second martingale · 7. Markov: only now matters · 8. When the path matters · 9. History · 10. This episode's question · 11. Exercises · 12. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 7,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)

## 2. Last episode's answer

Under $q$ the undiscounted share grows like the deposit: $E_Q[S_{n+1} \mid \mathcal{F}_n] = (0.55 \times 1.2 + 0.45 \times 0.8)\, S_n = 1.02\, S_n$. Not a martingale; only the discounted share is.

In [ ]:
out["prev"] = {"growth": 0.55 * 1.2 + 0.45 * 0.8}

## 3. A random walk

Switch to a fair coin and step $+1$ on heads, $-1$ on tails: $M_n = X_1 + \dots + X_n$. This **symmetric random walk** is the simplest random process there is, and (Part 2) Brownian motion is its limit.

In [ ]:
rng = np.random.default_rng(seed)
N = 10_000
walk = np.concatenate([[0], np.cumsum(rng.choice([-1, 1], N))])
every = 10
out["walk"] = {"steps": N, "end": int(walk[-1]), "end_abs": abs(int(walk[-1])), "end_positive": bool(walk[-1] > 0),
               "max": int(walk.max()), "min": int(walk.min()),
               "t": (np.arange(0, N + 1, every) / N).round(4).tolist(), "path": [walk[::every].tolist()],
               "ticks": [{"x": x, "label": f"{int(x * N):,}"} for x in (0, 0.25, 0.5, 0.75, 1)]}
assert walk[-1] > 0 and walk.min() < 0 < walk.max()      # the narration says "above where it started" and "as low as minus"
{k: v for k, v in out["walk"].items() if k not in ("t", "path", "ticks")}

## 4. Independent pieces add their variances

Each step has mean 0 and variance 1, and steps are independent, so $\mathrm{Var}(M_n) = n$, and any stretch of the walk, $M_k - M_l$, has variance $k - l$, whatever happened before it. 20,000 walks of 100 steps:

In [ ]:
W = np.cumsum(rng.choice([-1, 1], (20_000, 100)), axis=1)
end = W[:, -1]
k = np.arange(-40, 41, 2)
share = np.array([(end == x).mean() for x in k])
cx = np.linspace(-40, 40, 161)
cy = 2 * np.exp(-cx ** 2 / 200) / math.sqrt(2 * math.pi * 100)   # bars are 2 apart: 2 x the normal density
out["var"] = {"walks": 20_000, "n": 100, "var_100": float(end.var()), "var_last_50": float((end - W[:, 49]).var()),
              "k": k.tolist(), "share": share.round(5).tolist(), "curve_x": cx.round(2).tolist(), "curve_y": cy.round(5).tolist(),
              "x_range": [-42, 42], "ticks": [{"x": x, "label": f"{x:+d}" if x else "0"} for x in (-40, -20, 0, 20, 40)]}
assert abs(out["var"]["var_100"] / 100 - 1) < 0.03 and abs(out["var"]["var_last_50"] / 50 - 1) < 0.03
{k_: out["var"][k_] for k_ in ("var_100", "var_last_50")}

## 5. Squared steps, exactly

Every step squared is exactly 1, so the sum of squared steps over $n$ steps is exactly $n$, on every path, with no noise at all. Episode 1's "squared steps add up to the time elapsed", in its simplest form.

In [ ]:
qv = int((np.diff(walk) ** 2).sum())
assert qv == N
out["qv"] = {"steps": N, "sum": qv}

## 6. A second martingale

$M_n$ is a martingale (fair steps). So is $M_n^2 - n$: $E[M_{n+1}^2 \mid \mathcal{F}_n] = M_n^2 + 2M_n E[X_{n+1}] + E[X_{n+1}^2] = M_n^2 + 1$, and the $-n$ takes the 1 back off. In simulation its average stays near 0 at every time (within noise: the standard error at $n = 100$ is about $100\sqrt{2}/\sqrt{20000} \approx 1$).

In [ ]:
msq = [{"n": n, "mean": float((W[:, n - 1] ** 2 - n).mean()), "se": float((W[:, n - 1] ** 2 - n).std() / math.sqrt(20_000))} for n in (10, 50, 100)]
assert all(abs(r["mean"]) < 3 * r["se"] for r in msq)
out["msq"] = {"rows": msq}
pd.DataFrame(msq)

## 7. Markov: only now matters

Back to the coin world. A process is **Markov** if the future depends on the past only through the present. After HT and after TH the share is at 96 either way, and the best guess of $S_3$ is the same, $96 \times 1.04 = 99.84$: how it got to 96 doesn't matter.

In [ ]:
cw = CoinWorld()
S3 = lambda w: cw.price(w, 3)
g2 = cw.given(2, S3)
pathmax = lambda w: max(cw.path(w))
m2 = cw.given(2, pathmax)
assert abs(g2["HT"] - g2["TH"]) < 1e-12
assert abs(m2["HT"] - m2["TH"]) > 1
rows = cw.table(g2=lambda w: g2[w[:2]], max2=lambda w: max(cw.path(w)[:3]), m2=lambda w: m2[w[:2]], pathmax=pathmax)
out["coin"] = {"worlds": rows}
out["markov"] = {"S2": cw.price("HT", 2), "guess_HT": g2["HT"], "guess_TH": g2["TH"], "mid_worlds": ["HTH", "HTT", "THH", "THT"]}
out["lookback"] = {"max_so_far_HT": max(cw.path("HT")), "max_so_far_TH": max(cw.path("TH")), "guess_HT": m2["HT"], "guess_TH": m2["TH"]}
out["markov"], out["lookback"]

## 8. When the path matters

Now guess the **highest price along the path** (what a lookback option pays on). After HT the path has already been to 120; after TH it has only been to 100. Same price today, different best guesses: 120 vs 109.12. The share price is Markov, but to value a path-dependent payoff you must also track the running maximum.

In [ ]:
out["lookback"]

## 9. History

Andrey Markov published his first paper on chains of dependent variables, now called Markov chains, in 1906.

In [ ]:
out["history"] = {"markov": 1906}

## 10. This episode's question

A fair walk is at +3 after 50 steps. Best guess after 100? The remaining 50 steps average 0, so +3: the walk is a martingale.

In [ ]:
out["question"] = {"at": 3, "after": 50, "horizon": 100, "answer": 3}

## 11. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** What is $\mathrm{Var}(M_7 - M_3)$ for the fair ±1 walk?

<details><summary>Answer</summary>

Four independent steps, each with variance 1: $7 - 3 = 4$.

</details>

**Exercise 2 (By hand).** Show that $E[M_{n+1}^2 - (n+1) \mid \mathcal{F}_n] = M_n^2 - n$.

<details><summary>Answer</summary>

$M_{n+1}^2 = M_n^2 + 2M_n X_{n+1} + X_{n+1}^2$. Given $\mathcal{F}_n$: $M_n$ is known, $E[X_{n+1}] = 0$, $X_{n+1}^2 = 1$. So the guess is $M_n^2 + 1$, and subtracting $n + 1$ leaves $M_n^2 - n$.

</details>

**Exercise 3 (By hand).** Is the running maximum of the share, on its own, a Markov process?

<details><summary>Answer</summary>

No. Whether the maximum can rise next step depends on how far today's price is below it, which the maximum alone doesn't tell you. The pair (price, running maximum) is Markov.

</details>

**Exercise 4 (In code).** Simulate a walk with steps ±2. What is the variance after 100 steps, and the sum of squared steps?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
W2 = np.cumsum(rng.choice([-2, 2], (20_000, 100)), axis=1)
assert abs(W2[:, -1].var() / 400 - 1) < 0.03
print("variance after 100 steps:", round(W2[:, -1].var(), 1), "(theory 400); sum of squared steps: exactly", 100 * 4)

**Exercise 5 (In code).** Price a lookback call on the coin world that pays (highest price along the path − 100), using the risk-neutral odds q = 0.55 and 2% interest a step.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
look = lambda w: max(cw.path(w)) - 100
price = cw.expect(look, "Q") / 1.02 ** 3
assert price > cw.expect(lambda w: max(cw.price(w, 3) - 100, 0), "Q") / 1.02 ** 3   # worth more than the plain call
print(f"lookback price {price:.4f}")

## 12. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")